# Raw-сверка Voice ↔ CRM + история по ucpid

Расширение `omega_raw_match_export`. Два шага:

1. **Прямой матч**: `Id ПрПр` ↔ `productOfferId`/`key`, `Id задачи` ↔ `taskid`/`key` — как раньше (файлы 02/03).
2. **История по `ucpid`**: из совпавших CRM-строк берём `ucpid` клиента и подтягиваем ВСЕ его task/offer-строки из тех же `part-*.csv` (файл 05).

Не весь `part-*.csv` (Excel не откроет), а только по сматченным клиентам. Все id/ucpid пишутся как ТЕКСТ,
чтобы 19-значный `ucpid` не превращался в `2,17E+18` и находился поиском.

Для быстрой проверки одного звонка задайте `FOCUS_UCIDS = ["<ucid>"]`; пустой список = все клиенты.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import defaultdict
import ast, csv, re, warnings
from typing import Any
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 260)

## 1. Настройки

In [ ]:
BASE_DIR = Path.cwd()
VOICE_MATCH_XLSX = None          # напр. BASE_DIR / "Для метчинга.xlsx"; None = автопоиск
CRM_CSV_FILES = None             # напр. [BASE_DIR/"part-000.csv", ...]; None = все part-*.csv
FOCUS_UCIDS = ["55af3ede-5f3e-4e57-a75e-9c1b71ed976a"]   # [] = все сматченные клиенты
OUT_DIR = BASE_DIR / "omega_raw_match_with_history"
OUT_DIR.mkdir(exist_ok=True, parents=True)
print("Рабочая папка:", BASE_DIR, "| результат:", OUT_DIR, "| FOCUS:", FOCUS_UCIDS or "все")

## 2. Утилиты

In [ ]:
NULL_LIKE = {"", "NULL", "[NULL]", "None", "nan", "NaN", "NaT", "[]"}

def clean(v: Any) -> str:
    if v is None: return ""
    try:
        if pd.isna(v): return ""
    except Exception: pass
    return str(v).strip()

def canon_name(name: Any) -> str:
    s = clean(name).lower().replace("ё", "е")
    return re.sub(r"[\s_\-./()]+", "", s)

def find_col(df, exact=None, regex=None, required=False, label=""):
    exact = exact or []; regex = regex or []
    exact_canons = {canon_name(x) for x in exact}
    for col in df.columns:
        if canon_name(col) in exact_canons: return col
    for pattern in regex:
        rx = re.compile(pattern, flags=re.I)
        for col in df.columns:
            if rx.search(canon_name(col)) or rx.search(clean(col)): return col
    if required:
        raise KeyError(f"Не найдена колонка {label or exact or regex}. Есть: {list(df.columns)}")
    return None

def strip_wrappers(v: Any) -> str:
    t = clean(v)
    for _ in range(4):
        old = t
        t = t.strip().strip("'").strip('"').strip()
        if t.startswith("[") and t.endswith("]"): t = t[1:-1].strip()
        if t.startswith("(") and t.endswith(")"): t = t[1:-1].strip()
        if t == old: break
    return t

def explode_cell(v: Any) -> list[str]:
    t = clean(v)
    if not t or t in NULL_LIKE: return []
    if (t.startswith("[") and t.endswith("]")) or (t.startswith("(") and t.endswith(")")):
        try:
            parsed = ast.literal_eval(t)
            if isinstance(parsed, (list, tuple, set)):
                out = []
                for it in parsed: out.extend(explode_cell(it))
                return out
            return explode_cell(parsed)
        except Exception: pass
    t = strip_wrappers(t)
    if not t or t in NULL_LIKE: return []
    if "," in t:
        return [strip_wrappers(x) for x in t.split(",") if strip_wrappers(x) and strip_wrappers(x) not in NULL_LIKE]
    return [t]

def norm_key(v: Any) -> str:
    t = strip_wrappers(v)
    if not t or t in NULL_LIKE: return ""
    return re.sub(r"\s+", "", t).upper()

def key_variants(v: Any) -> list[str]:
    out = []
    for part in explode_cell(v):
        k = norm_key(part)
        if k: out.append(k)
    return list(dict.fromkeys(out))

def write_xlsx_text(path, sheets):
    from openpyxl.utils import get_column_letter
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet, df in sheets.items():
            safe = sheet[:31]
            (df if len(df) else pd.DataFrame({"note": ["пусто"]})).to_excel(writer, sheet_name=safe, index=False)
            ws = writer.book[safe]
            ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
            for ci in range(1, ws.max_column + 1):
                L = get_column_letter(ci)
                for cell in ws[L]:
                    cell.number_format = "@"     # ТЕКСТ: ucpid/id не станут 2,17E+18
                header = str(ws.cell(1, ci).value or "")
                ws.column_dimensions[L].width = min(max(len(header) + 2, 12), 60)
    print("saved:", path)

## 3. Voice-ключи

In [ ]:
def find_voice_match_file() -> Path:
    if VOICE_MATCH_XLSX is not None: return Path(VOICE_MATCH_XLSX)
    cands = [p for p in BASE_DIR.glob("*.xlsx") if not p.name.startswith("~$") and ("метч" in p.name.lower() or "match" in p.name.lower())]
    if not cands: raise FileNotFoundError("Нет Excel с ключами Voice (напр. 'Для метчинга.xlsx').")
    exact = [p for p in cands if p.name == "Для метчинга.xlsx"]
    return exact[0] if exact else max(cands, key=lambda p: p.stat().st_mtime)

voice = pd.read_excel(find_voice_match_file(), dtype=str)
vc = {
    "ucid": find_col(voice, exact=["ucid"], required=True, label="ucid"),
    "task_id": find_col(voice, exact=["Id задачи"], regex=[r"(id|ид).*задач", r"task.*id"], required=True, label="Id задачи"),
    "offer_id": find_col(voice, exact=["Id ПрПр"], regex=[r"(id|ид).*прпр", r"product.*offer"], required=True, label="Id ПрПр"),
    "org_id": find_col(voice, exact=["Id Организации (crm/ЕКП)", "Id Организации"], regex=[r"(id|ид).*орган", r"ucp", r"екп"]),
    "date": find_col(voice, exact=["Дата активности", "Дата звонка"]),
}
voice_small = pd.DataFrame({
    "ucid": voice[vc["ucid"]].map(clean),
    "Id задачи": voice[vc["task_id"]].map(clean),
    "Id ПрПр": voice[vc["offer_id"]].map(clean),
    "Id Организации": voice[vc["org_id"]].map(clean) if vc["org_id"] else "",
    "Дата активности": voice[vc["date"]].map(clean) if vc["date"] else "",
})
# ключ -> множество ucid; и дата звонка по ucid
key_to_ucids = defaultdict(set); call_date = {}
task_key_set, offer_key_set = set(), set()
for _, r in voice_small.iterrows():
    call_date[r["ucid"]] = r["Дата активности"]
    for k in key_variants(r["Id задачи"]): task_key_set.add(k); key_to_ucids[k].add(r["ucid"])
    for k in key_variants(r["Id ПрПр"]):  offer_key_set.add(k); key_to_ucids[k].add(r["ucid"])
print("Voice строк:", len(voice_small), "| task-ключей:", len(task_key_set), "| offer-ключей:", len(offer_key_set))

## 4. Читаем part-*.csv

In [ ]:
def read_crm(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return pd.read_csv(path, sep="\t", encoding="cp1251", engine="python",
                           quoting=csv.QUOTE_NONE, on_bad_lines="warn", dtype=str)

crm_paths = [Path(p) for p in CRM_CSV_FILES] if CRM_CSV_FILES else sorted(BASE_DIR.glob("part-*.csv"))
if not crm_paths: raise FileNotFoundError("Нет part-*.csv рядом с ноутбуком.")
crm_frames = {}
for p in crm_paths:
    crm_frames[p.name] = read_crm(p); print(p.name, crm_frames[p.name].shape)

## 5. Профиль файлов (taskid / productOfferId / key / ucpid)

In [ ]:
def profile(name, df):
    return {
        "file": name, "rows": len(df),
        "taskid_col": find_col(df, exact=["taskid"], regex=[r"^taskid$", r"task.*id"]) or "",
        "productOfferId_col": find_col(df, exact=["productOfferId"], regex=[r"product.*offer.*id"]) or "",
        "key_col": find_col(df, exact=["key"], regex=[r"^key$"]) or "",
        "ucpid_col": find_col(df, exact=["ucpid"], regex=[r"^ucpid$"]) or "",
        "date_col": find_col(df, exact=["factEndDate", "endDate", "updateTime", "creationTime"], regex=[r"end.?date", r"update.?time", r"creation.?time"]) or "",
    }

crm_profiles = pd.DataFrame([profile(n, d) for n, d in crm_frames.items()])
crm_profiles["entity"] = crm_profiles.apply(lambda r: "task" if r["taskid_col"] else ("offer" if r["productOfferId_col"] else "unknown"), axis=1)
display(crm_profiles)
if not crm_profiles["ucpid_col"].ne("").any():
    raise ValueError("Ни в одном part-файле не найден столбец ucpid — история по ucpid невозможна.")

## 6. Шаг 1 — прямой матч (файлы 02/03, как раньше) и сбор ucpid клиентов

In [ ]:
ucpid_to_ucids = defaultdict(set)       # ucpid -> {ucid}
direct_rows = {"task": [], "offer": []}  # прямо совпавшие сырые строки

for _, prof in crm_profiles.iterrows():
    df = crm_frames[prof["file"]]
    ucpid_col = prof["ucpid_col"]
    idcols = [c for c in [prof["taskid_col"], prof["productOfferId_col"], prof["key_col"]] if c]
    if not idcols: continue
    matched_idx = []
    for idx, row in df.iterrows():
        rkeys = set()
        for c in idcols: rkeys |= set(key_variants(row.get(c)))
        hit = rkeys & (task_key_set | offer_key_set)
        if hit:
            matched_idx.append(idx)
            up = norm_key(row.get(ucpid_col)) if ucpid_col else ""
            if up:
                for k in hit: ucpid_to_ucids[up] |= key_to_ucids.get(k, set())
    if matched_idx and prof["entity"] in direct_rows:
        direct_rows[prof["entity"]].append(df.loc[matched_idx].copy())

# Voice «Id Организации» — по бриджу это тоже ucpid; добавим как запасной источник
for _, r in voice_small.iterrows():
    for k in key_variants(r["Id Организации"]):
        if k: ucpid_to_ucids[k].add(r["ucid"])

if FOCUS_UCIDS:
    focus = set(FOCUS_UCIDS)
    ucpid_to_ucids = {up: us for up, us in ucpid_to_ucids.items() if us & focus}
matched_ucpids = set(ucpid_to_ucids)
print("Сматченных клиентов (ucpid):", len(matched_ucpids))
print("Прямых task-строк:", sum(len(x) for x in direct_rows["task"]), "| offer-строк:", sum(len(x) for x in direct_rows["offer"]))

## 7. Шаг 2 — вся история по ucpid (файл 05)

In [ ]:
def pull_history(entity):
    parts = []
    for _, prof in crm_profiles[crm_profiles["entity"].eq(entity)].iterrows():
        df = crm_frames[prof["file"]]; ucpid_col = prof["ucpid_col"]
        if not ucpid_col: continue
        m = df[df[ucpid_col].map(norm_key).isin(matched_ucpids)].copy()
        if not len(m): continue
        idcol = prof["taskid_col"] or prof["productOfferId_col"] or prof["key_col"]
        ins = {
            "_ucpid": m[ucpid_col].map(norm_key),
            "_ucids": m[ucpid_col].map(lambda u: "; ".join(sorted(ucpid_to_ucids.get(norm_key(u), [])))),
            "_entity": entity, "_src_file": prof["file"], "_src_row": m.index,
            "_is_matched_object": m[idcol].map(norm_key).isin(task_key_set | offer_key_set) if idcol else False,
        }
        if prof["date_col"]:
            ins["_event_date"] = pd.to_datetime(m[prof["date_col"]], errors="coerce").dt.strftime("%Y-%m-%d %H:%M")
        for i, (name, val) in enumerate(ins.items()): m.insert(i, name, val)
        parts.append(m)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

task_hist = pull_history("task")
offer_hist = pull_history("offer")
if len(task_hist) and "_event_date" in task_hist: task_hist = task_hist.sort_values(["_ucids", "_event_date"])
if len(offer_hist) and "_event_date" in offer_hist: offer_hist = offer_hist.sort_values(["_ucids", "_event_date"])
print("История: task-строк", len(task_hist), "| offer-строк", len(offer_hist))

ucpid_map = pd.DataFrame(
    [{"ucpid": up, "ucids": "; ".join(sorted(us)), "n_calls": len(us)} for up, us in sorted(ucpid_to_ucids.items())]
)

## 8. Сохраняем Excel

In [ ]:
task_direct = pd.concat(direct_rows["task"], ignore_index=True) if direct_rows["task"] else pd.DataFrame()
offer_direct = pd.concat(direct_rows["offer"], ignore_index=True) if direct_rows["offer"] else pd.DataFrame()

write_xlsx_text(OUT_DIR / "02_crm_task_rows_by_voice_id_task.xlsx", {"CRM task rows (direct)": task_direct})
write_xlsx_text(OUT_DIR / "03_crm_offer_rows_by_voice_id_prpr.xlsx", {"CRM offer rows (direct)": offer_direct})
write_xlsx_text(OUT_DIR / "05_crm_history_by_ucpid.xlsx", {
    "task history": task_hist,
    "offer history": offer_hist,
    "ucpid to ucid": ucpid_map,
})
print("Готово. Смотри 05_crm_history_by_ucpid.xlsx")

## Как проверять глазами

1. `05_crm_history_by_ucpid.xlsx` → лист **«offer history»** / **«task history»**.
2. Отфильтруй по `_ucids` (конкретный ucid звонка) или по `_ucpid` — увидишь ВСЮ историю клиента.
3. `_is_matched_object = True` — та самая строка, по которой звонок сматчился (объект звонка).
4. `_event_date` — для отсечки «до/после звонка» сравни с датой звонка из `ucpid to ucid` / Voice.
5. Все id и ucpid — текст, поэтому Ctrl+F по ним работает (в т.ч. по 19-значному ucpid).

Файлы 02/03 остаются как прямой матч (одна строка на попадание) — для сверки самого факта склейки.